# STEP5. 최종 결과와 지원연계

> **Takeaway.** 핵심 신호부터 현장 확인 질문과 기존 기관 인계까지 한 흐름으로 제시한다.

## 분석 질문

어느 업종을 먼저 확인하고 무엇을 확인한 뒤 어디로 인계할 것인가?

## 입력

| 구분 | 설정 변수 |
|---|---|
| 분류 결과 | `config.TRIAGE_PANEL_PATH` |
| 설명 근거 | `config.HANDOFF_TRACE_PATH` |

## 출력

| 구분 | 설정 변수 |
|---|---|
| 인계 카드 | `config.HANDOFF_CARD_PATH` |
| 보고 자산 | `config.REPORT_ASSET_DIR` |

## 전체 흐름

`핵심 신호 → 1차 분류 → 선택적 재검토 → 외부 근거 → 설명·인계`

## 창원국가산단 산업·고용 전환진단 및 지원연계 모형

## 1. 문제 정의

**기존 산업동향 분석을 기업·현장 확인의 표준화된 진입점으로 전환한다.**

창원에는 산업동향·위기대응·기업진단·고용·훈련지원 기능이 이미 존재한다. 기관별 분석단위와 업무 진입경로가 다르므로, 분산된 기존 체계 사이의 연결 공백을 보완하는 **업종 단위 의사결정 지원모형**을 제안한다.

공개자료 확인 범위에서 표준 연결 사례를 확인하기 어렵다는 뜻이며, 실제 협업이 없다는 주장은 아니다.

모형은 이번 분기에 어느 업종을 어느 수준으로 먼저 확인할지만 결정한다. 위기 여부·확률·원인·기업별 지원·사업 선정·예산은 결정하지 않는다.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, HTML, Markdown
ROOT = Path.cwd() if (Path.cwd()/'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT/'src'))
from handoff import delivery
TRIAGE=ROOT/'outputs/final_model/02_triage/tables'
HANDOFF=ROOT/'outputs/final_model/05_handoff/tables'
VALIDATION_ROOT=ROOT/'logs/validation'
VALIDATION=VALIDATION_ROOT/'triage'
REPORT_ASSETS=ROOT/'outputs/final_model/06_report_assets'
PATHS={'decision_panel.csv':TRIAGE/'triage_panel.csv','decision_latest.csv':TRIAGE/'triage_latest_full.csv','rule_provenance.csv':TRIAGE/'triage_rule_provenance.csv','stage_distribution_by_industry.csv':TRIAGE/'triage_distribution_by_industry.csv','stage_distribution_by_quarter.csv':TRIAGE/'triage_distribution_by_quarter.csv','diagnostic_cards_latest.csv':HANDOFF/'handoff_cards_latest.csv','institution_handoff_map.csv':HANDOFF/'institution_routing_map.csv'}
def read(name): return pd.read_csv(PATHS.get(name, VALIDATION/name))
panel=read('decision_panel.csv')
latest=read('decision_latest.csv')
assert len(panel)==180 and len(latest)==10
pd.set_option('display.max_columns', 20)
from IPython.display import display as _display
def display(value):
    if isinstance(value, pd.DataFrame):
        markup=value.to_html(index=False,render_links=True,na_rep='미확인',border=0)
        if len(value)>30:
            markup='<details><summary>전체 '+str(len(value))+'행 펼치기</summary>'+markup+'</details>'
        _display(HTML('<div style="overflow:auto;max-width:100%">'+markup+'</div>'))
    else:
        _display(value)


## 2. 기존 창원의 관련 체계

기존 기관과 기능을 전제로 한다. 링크는 확인한 공개자료이며 담당기관 배정·사업 이용 가능성은 현장 인계 시 재확인한다.

In [2]:
display(read('institution_handoff_map.csv'))

role,institution,unit,function,source_url
산업·경제동향,창원상공회의소,지역·산단·업종,정기 경제동향,https://changwoncci.korcham.net/front/board/boardContentsView.do?boardId=11175&contId=130203&menuId=3992
산업·고용동향,창원산업진흥원,지역·산업,산업·경제동향 및 고용동향 자료,https://www.cwip.or.kr/
지역 위기진단,경남TP 위기지원센터,중소기업 밀집지역·기업,모니터링·심층현장조사·맞춤지원,https://www.gntp.or.kr/introduce/staff
기업 진단·고용지원,창원고용복지+센터,기업·구직자,기업 도약보장 패키지 등 진단·연계(운영 공지 시점 확인 필요),https://www.moel.go.kr/local/changwon/news/reportexplan/view.do?bbs_seq=20240901144
기업 현장지원,창원산업진흥원 기업지원 기능·마이스터센터,기업,현장애로 컨설팅·기술지원,https://www.cwip.or.kr/
인력·훈련 연계,경남지역인적자원개발위원회,산업·기업 인력수요,수요조사·맞춤형 훈련 연계,https://gnhrd.or.kr/


## 3. 연결 공백

**어느 업종을 먼저 확인하고, 무엇을 확인한 뒤, 어느 기존 기능으로 넘길 것인가?**

산업동향은 지역·산단·업종, 위기대응은 밀집지역, 기업지원은 기업, 고용·훈련은 기업·근로자·산업수요 단위다. 이 사이를 잇는 근거와 질문을 표준화한다.

## 4. 우리가 보완하는 영역

In [3]:
display(HTML(delivery.flow_html()))

## 5. 데이터와 독립 재현

로컬 KICOX 원자료 → 제조업 10개 업종 master → Q1~Q3·E/R/A/P → 2022Q1~2026Q2 180행. 명목 생산은 분기 합계, 고용은 분기말. 반복신호는 분석 시작 이전 분기도 사용한다. 원자료 재구축·독립 scalar 검산은 기존 ELECTRE 출력을 덮어쓰지 않는다.

In [4]:
display(pd.DataFrame(json.loads((VALIDATION_ROOT/'final_qa/raw_rebuild_verification.json').read_text(encoding='utf-8'))))
display(json.loads((VALIDATION/'audit_v3/verification.json').read_text(encoding='utf-8')))

file,rows,match
changwon_industry_master,340,True
changwon_total_master,34,True
changwon_state_panel,180,True


{'historical_reference_rows': 180,
 'independent_scalar_axes_match': True,
 'independent_scalar_stages_match': True,
 'original_v3_distribution': {'관찰': 113, '추가확인': 32, '자료확인': 20, '우선점검': 15},
 'final_distribution': {'관찰': 128, '추가확인': 35, '우선점검': 17},
 'baseline_to_final_changes': 21}

## 6. Q1 상태 — 무엇을 확인할 것인가

상태가 단계를 직접 결정하지 않는다. S1 확장·인력수급, S2 자동화·외주·미충원, S3 선행채용·생산 일시성, S4 수주·감산·고용조정 가능성을 **질문**으로 넘긴다. 원인 확정이 아니다.

In [5]:
display(latest[['industry','q1_state','q1_state_label','q1_question_route']])

industry,q1_state,q1_state_label,q1_question_route
기계,S4,생산↓·고용↓,수주잔량·가동률·휴업/감산·기업 수 변동·고용조정 계획 신고 여부를 확인한다(원인 후보이며 입증된 원인이 아님).
목재종이,S4,생산↓·고용↓,수주잔량·가동률·휴업/감산·기업 수 변동·고용조정 계획 신고 여부를 확인한다(원인 후보이며 입증된 원인이 아님).
운송장비,S2,생산↑·고용↓,"자동화·설비투자, 외주·도급 전환, 미충원·숙련 불일치, 직무구조 변화 가능성을 질문으로 확인한다(원인 후보이며 입증된 원인이 아님)."
전기전자,S2,생산↑·고용↓,"자동화·설비투자, 외주·도급 전환, 미충원·숙련 불일치, 직무구조 변화 가능성을 질문으로 확인한다(원인 후보이며 입증된 원인이 아님)."
철강,S2,생산↑·고용↓,"자동화·설비투자, 외주·도급 전환, 미충원·숙련 불일치, 직무구조 변화 가능성을 질문으로 확인한다(원인 후보이며 입증된 원인이 아님)."
음식료,S2,생산↑·고용↓,"자동화·설비투자, 외주·도급 전환, 미충원·숙련 불일치, 직무구조 변화 가능성을 질문으로 확인한다(원인 후보이며 입증된 원인이 아님)."
비금속,S2,생산↑·고용↓,"자동화·설비투자, 외주·도급 전환, 미충원·숙련 불일치, 직무구조 변화 가능성을 질문으로 확인한다(원인 후보이며 입증된 원인이 아님)."
기타,S1,생산↑·고용↑,미충원·숙련수요·증가의 지속가능성을 확인한다.
석유화학,S3,생산↓·고용↑,선제채용·신규 입주기업·고용조정 시차·생산 단가와 물량 구분 가능성을 확인한다.
섬유의복,N,한 축 이상 정확한 0,"원자료상 정확한 0인지 반올림·개정 영향인지, 0이 아닌 축의 방향과 규모를 확인한다."


## 7. Q2 규모 — 얼마나 큰 영향인가

고용인원 변화·비중·순증감 기여율과 같은 단계 내 처리순서를 전달한다. 기여율은 전체 순변화 상쇄가 크면 미표시한다. A는 현재 산단 전체 고용 대비 감소인원으로 별도 정의한 규모효과다. Q2 전체 점수를 만들지 않는다.

In [6]:
display(latest[['industry','q2_employment_delta','q2_employment_yoy','q2_employment_share','q2_contribution','q2_same_stage_rank']].round(3))

industry,q2_employment_delta,q2_employment_yoy,q2_employment_share,q2_contribution,q2_same_stage_rank
기계,-3979.0,-6.340,51.192,91.851,1
목재종이,-48.0,-10.084,0.373,1.108,2
운송장비,-129.0,-0.785,14.202,2.978,1
전기전자,-116.0,-0.413,24.366,2.678,2
철강,-112.0,-1.143,8.438,2.585,3
음식료,-25.0,-3.788,0.553,0.577,4
비금속,-2.0,-1.081,0.159,0.046,5
기타,25.0,17.123,0.149,-0.577,6
석유화학,54.0,9.558,0.539,-1.247,7
섬유의복,0.0,0.000,0.030,-0.000,8


## 8. Q3 시간 — 반복·지속 여부

**Q3 시간축에서 도출한 반복·지속 신호를 상위 점검단계의 보강조건으로 활용하였다.**

현재와 직전 달력분기의 고용축 진입신호다. 동일 상태 지속기간이나 음의 고용 YoY 연속길이 g4와 다르다. 생산 결측 때 Q1 INVALID는 그대로 남는다.

In [7]:
display(read('q3_definition_comparison.csv').tail(20))

industry,quarter,q3_state_run_length,q3_transition,q3_repeated_signal,g4_delta00
전기전자,2026Q1,1.0,S2 → S4,False,5
전기전자,2026Q2,1.0,S4 → S2,False,6
철강,2022Q1,1.0,이전 또는 현재 상태 미확인,False,0
철강,2022Q2,2.0,S1 → S1,False,0
철강,2022Q3,3.0,S1 → S1,False,0
철강,2022Q4,1.0,S1 → S4,False,1
철강,2023Q1,2.0,S4 → S4,False,2
철강,2023Q2,3.0,S4 → S4,False,3
철강,2023Q3,4.0,S4 → S4,False,4
철강,2023Q4,미확인,이전 또는 현재 상태 미확인,False,0


## 9. 분석과 판단엔진의 역할

|분석|역할|
|---|---|
|Q1|확인질문·담당 기능|
|Q2|영향 규모·같은 단계 처리순서; A에 감소인원 반영|
|Q3 시간축|반복 고용진입신호 보강|
|E/R/A 및 P·반복·규모 조건|점검단계 판단|

## 10. 최종 트리아지 규칙

고용 핵심자료가 없으면 판정보류(자료확인). 그 외 **상위 고용신호 AND (생산 보강 OR 반복) AND 300인 이상 → 우선점검**. 이 조건에 못 미치더라도 진입신호가 있으면 추가확인, 나머지는 관찰. P만 없으면 알려진 신호로 최소판정하고 자료 플래그를 인계한다. 관찰도 다음 분기 재점검한다.

## 11. 출처와 프로젝트 운영규칙

법정 지정기준을 직접 적용하지 않는다. 같은 숫자도 대상·기간·목적이 달라 DIRECT_OFFICIAL은 없다. R 상위 10%p와 A 1/2는 자체 운영규칙이다.

In [8]:
display(read('rule_provenance.csv'))

rule_name,definition,threshold,role,source_category,source_url,source_note,reason_for_adaptation,checked_at
E,"max(0, -KICOX 분기말 고용 YoY)",5% / 10%,고용 진입·상위 신호,PRINCIPLE_ADAPTED,https://www.law.go.kr/LSW/admRulInfoP.do?admRulSeq=2100000278624&chrClsCd=010201,"2026.5.4 시행 고용위기지역 고시 제3조①2호 5%, ④2호 10% 확인. 지역 피보험자 최근 6개월 평균을 업종 종사자 분기말 YoY로 변경. 다른 법정 결합요건·심의절차를 재현하지 않음.","2026.5.4 시행 고용위기지역 고시 제3조①2호 5%, ④2호 10% 확인. 지역 피보험자 최근 6개월 평균을 업종 종사자 분기말 YoY로 변경. 다른 법정 결합요건·심의절차를 재현하지 않음.",2026-09-18
R_entry,"max(0, 산단 제조업 전체 YoY - 업종 YoY)",5%p,상대 열위 진입,PRINCIPLE_ADAPTED,https://m.work24.go.kr/cm/c/f/1100/selecSystInfo.do?systClId=SC00000364&systCnntId=CI00001365&systId=SI00000375,공식 비교는 모든 업종 피보험자 평균. 본 모형은 동일 산단 제조업 10개 업종 합계와 비교.,공식 비교는 모든 업종 피보험자 평균. 본 모형은 동일 산단 제조업 10개 업종 합계와 비교.,2026-09-18
R_up,R과 동일,10%p,상대 열위 상위,PROJECT_OPERATIONAL,미확인,10%p 상위 경계의 직접 조문 근거 없음. 진입 경계와 출처 분리.,10%p 상위 경계의 직접 조문 근거 없음. 진입 경계와 출처 분리.,2026-09-18
A,"max(0, 전년동기 고용 - 현재 고용) / 현재 산단 제조업 고용 × 100",1% / 2%,규모효과 진입·상위,PROJECT_OPERATIONAL,미확인,현재 산단 고용 100명당 전년동기 대비 감소인원 1/2명에 해당. 최적값·정책기준 아님. 대형 업종 영향 큼.,현재 산단 고용 100명당 전년동기 대비 감소인원 1/2명에 해당. 최적값·정책기준 아님. 대형 업종 영향 큼.,2026-09-18
P,"max(0, -명목 분기 생산액 YoY)",5%,상위단계 보강,PRINCIPLE_ADAPTED,https://www.law.go.kr/LSW/admRulLsInfoP.do?admRulSeq=2100000255684,현행 제2조는 전년 및 전전년 동기 생산액/생산량 등 결합요건. 본 모형은 명목 생산액 전년동기 비교만 사용하며 가격·물량 분해 불가.,현행 제2조는 전년 및 전전년 동기 생산액/생산량 등 결합요건. 본 모형은 명목 생산액 전년동기 비교만 사용하며 가격·물량 분해 불가.,2026-09-18
repeated_employment_entry_signal,현재 및 실제 직전 달력분기에 E/R/A 진입신호 존재,연속 2분기,Q3 시간축 보강,PRINCIPLE_ADAPTED,https://www.law.go.kr/LSW/admRulLsInfoP.do?admRulSeq=2100000209796,"지속성 원리 참고. 고시 월별 지속요건, 기존 g4 음의 고용 YoY 연속길이, 동일 Q1 상태 run_length와 모두 다름.","지속성 원리 참고. 고시 월별 지속요건, 기존 g4 음의 고용 YoY 연속길이, 동일 Q1 상태 run_length와 모두 다름.",2026-09-18
scale_gate,현재 업종 고용이 기준 이상,300인,우선점검 진입만 제한,PRINCIPLE_ADAPTED,https://www.law.go.kr/LSW/admRulLsInfoP.do?admRulSeq=2100000255684,공식 주된 산업의 비중·종사자 요건 중 규모 원리만 참고한 운영규칙. 추가확인 신호 보존. 300인 최적성 미입증.,공식 주된 산업의 비중·종사자 요건 중 규모 원리만 참고한 운영규칙. 추가확인 신호 보존. 300인 최적성 미입증.,2026-09-18
ladder,상위 고용신호 AND (P OR 반복) AND 규모; 아니면 진입신호,관찰 / 추가확인 / 우선점검,행동단계,PROJECT_OPERATIONAL,미확인,법정 지정의 결합식이 아니라 내부 점검 사다리.,법정 지정의 결합식이 아니라 내부 점검 사다리.,2026-09-18
missing,핵심 E/R/A·고용 미확인 시 판정보류; P만 없으면 알려진 신호로 최소판정,보간 없음,자료품질,PROJECT_OPERATIONAL,미확인,미확인 보강조건은 충족으로 간주하지 않음. 최소판정과 자료 플래그 함께 전달.,미확인 보강조건은 충족으로 간주하지 않음. 최소판정과 자료 플래그 함께 전달.,2026-09-18
same_stage_rank,"같은 분기·단계에서 감소인원 내림차순, 동률 업종명",감소인원,처리순서,PROJECT_OPERATIONAL,미확인,Q2 전체 점수화 아님. 동률 순서는 위험 우열이 아닌 결정적 표시순서.,Q2 전체 점수화 아님. 동률 순서는 위험 우열이 아닌 결정적 표시순서.,2026-09-18


## 12. 핵심 민감도와 감사

원본 Q3 제거는 6행, 수정판은 8행이다. 생산결측 20행 처리와 분석창 이전 정보 복구를 분리했다. A 2/4에서 최신 기계, 규모 500인에서 최신 목재종이가 추가확인으로 바뀐다. 핵심 대상의 **우선점검 단계는 운영규칙에 조건부**이며 최적성·예측 정확도를 주장하지 않는다. 1/2 사전 선택의 증거는 확보하지 못했다.

In [9]:
display(read('sensitivity_own_rules.csv'))
display(read('missingness_summary.csv').query('data_quality_production_missing'))
display(read('history_boundary_changed_rows.csv'))
display(read('q3_decisive_rows.csv')[['industry','quarter','stage','stage_reason']])

variant,changed_rows,priority_changed_rows,latest_changed_rows,우선점검,추가확인,관찰,자료확인,규모미달
A 0.5/1,5,2,0,19,36,125,0,0
A 1/2,0,0,0,17,35,128,0,0
A 2/4,6,6,1,11,41,128,0,0
A 제거,9,9,1,8,40,132,0,0
규모 200,4,4,0,21,31,128,0,0
규모 300,0,0,0,17,35,128,0,0
규모 500,2,2,1,15,37,128,0,0
규모 1000,8,8,1,9,43,128,0,0
gate 없음,24,24,0,41,11,128,0,0
hard exclusion,66,0,3,17,4,93,0,66


quarter,data_quality_core_missing,data_quality_production_missing,complete_stage,stage,rows
2023Q4,False,True,자료확인,관찰,7
2023Q4,False,True,자료확인,우선점검,1
2023Q4,False,True,자료확인,추가확인,2
2024Q4,False,True,자료확인,관찰,8
2024Q4,False,True,자료확인,추가확인,2


industry,quarter,E,R,A,P,persist,stage,stage_reason
기계,2022Q1,3.671835,3.911123,2.041025,0.0,True,우선점검,산단 제조업 고용의 2.04% 감소(상위경계 2.0% 통과) → 보강: 직전 분기에도 진입신호(지속)


industry,quarter,stage,stage_reason
기계,2022Q1,우선점검,산단 제조업 고용의 2.04% 감소(상위경계 2.0% 통과) → 보강: 직전 분기에도 진입신호(지속)
기계,2022Q2,우선점검,산단 제조업 고용의 2.16% 감소(상위경계 2.0% 통과) → 보강: 직전 분기에도 진입신호(지속)
기계,2022Q3,우선점검,산단 제조업 고용의 2.10% 감소(상위경계 2.0% 통과) → 보강: 직전 분기에도 진입신호(지속)
기계,2023Q1,우선점검,고용감소 9.7%(진입경계 5% 통과) · 산단평균 대비 6.7%p 열위(진입경계 통과) · 산단 제조업 고용의 5.38% 감소(상위경계 2.0% 통과) → 보강: 직전 분기에도 진입신호(지속)
기계,2023Q2,우선점검,고용감소 8.8%(진입경계 5% 통과) · 산단평균 대비 5.8%p 열위(진입경계 통과) · 산단 제조업 고용의 4.82% 감소(상위경계 2.0% 통과) → 보강: 직전 분기에도 진입신호(지속)
기계,2023Q3,우선점검,고용감소 8.4%(진입경계 5% 통과) · 산단평균 대비 6.5%p 열위(진입경계 통과) · 산단 제조업 고용의 4.54% 감소(상위경계 2.0% 통과) → 보강: 직전 분기에도 진입신호(지속)
기계,2023Q4,우선점검,고용감소 6.8%(진입경계 5% 통과) · 산단평균 대비 7.8%p 열위(진입경계 통과) · 산단 제조업 고용의 3.61% 감소(상위경계 2.0% 통과) → 보강: 직전 분기에도 진입신호(지속) [생산자료 미확인; 확인 가능한 신호에 따른 최소판정]
음식료,2026Q1,우선점검,고용감소 21.5%(상위경계 10% 통과) · 산단평균 대비 19.8%p 열위(상위경계 통과) → 보강: 직전 분기에도 진입신호(지속)


## 13. 180행 전체 결과

색과 단계명을 함께 표시한다. 각 행의 계산·근거·품질 플래그는 decision_panel.csv에 있다.

In [10]:
display(HTML(delivery.stage_grid(panel)))
display(panel[['industry','quarter','decision_stage','decision_entry_trigger','decision_reinforcement','decision_scale_gate','decision_reason']])

quarter,2022Q1,2022Q2,2022Q3,2022Q4,2023Q1,2023Q2,2023Q3,2023Q4,2024Q1,2024Q2,2024Q3,2024Q4,2025Q1,2025Q2,2025Q3,2025Q4,2026Q1,2026Q2
industry,,,,,,,,,,,,,,,,,,
기계,우선점검,우선점검,우선점검,우선점검,우선점검,우선점검,우선점검,우선점검,관찰,관찰,관찰,관찰,관찰,관찰,관찰,관찰,관찰,우선점검
기타,관찰,관찰,관찰,추가확인,추가확인,추가확인,추가확인,관찰,관찰,관찰,관찰,관찰,추가확인,추가확인,추가확인,추가확인,관찰,관찰
목재종이,관찰,관찰,관찰,관찰,관찰,관찰,추가확인,추가확인,추가확인,추가확인,추가확인,추가확인,관찰,관찰,관찰,관찰,우선점검,우선점검
비금속,관찰,관찰,관찰,관찰,관찰,관찰,관찰,관찰,관찰,추가확인,추가확인,관찰,추가확인,추가확인,추가확인,추가확인,관찰,관찰
석유화학,관찰,관찰,관찰,관찰,관찰,관찰,관찰,관찰,관찰,관찰,관찰,관찰,관찰,우선점검,관찰,관찰,관찰,관찰
섬유의복,추가확인,추가확인,추가확인,관찰,관찰,관찰,관찰,추가확인,추가확인,추가확인,추가확인,추가확인,추가확인,추가확인,추가확인,관찰,관찰,관찰
운송장비,관찰,관찰,관찰,관찰,관찰,관찰,관찰,관찰,관찰,관찰,관찰,관찰,관찰,관찰,관찰,추가확인,관찰,관찰
음식료,관찰,관찰,관찰,관찰,관찰,관찰,관찰,관찰,관찰,관찰,관찰,관찰,우선점검,우선점검,우선점검,우선점검,우선점검,관찰
전기전자,관찰,관찰,관찰,관찰,관찰,관찰,관찰,관찰,관찰,관찰,관찰,관찰,추가확인,추가확인,관찰,추가확인,관찰,관찰


industry,quarter,decision_stage,decision_entry_trigger,decision_reinforcement,decision_scale_gate,decision_reason
기계,2022Q1,우선점검,A,반복,True,산단 제조업 고용의 2.04% 감소(상위경계 2.0% 통과) → 보강: 직전 분기에도 진입신호(지속)
기타,2022Q1,관찰,없음,생산,False,고용 축 진입신호 없음 (생산 11.2% 단독 감소 — 확인질문에 반영)
목재종이,2022Q1,관찰,없음,확인된 보강 없음,False,고용 축 진입신호 없음
비금속,2022Q1,관찰,없음,확인된 보강 없음,False,고용 축 진입신호 없음
석유화학,2022Q1,관찰,없음,확인된 보강 없음,True,고용 축 진입신호 없음
섬유의복,2022Q1,추가확인,E|R,반복,False,고용감소 11.4%(상위경계 10% 통과) · 산단평균 대비 11.6%p 열위(상위경계 통과) [상위경계 통과했으나 고용 39인 < 300인으로 우선점검 제외]
운송장비,2022Q1,관찰,없음,확인된 보강 없음,True,고용 축 진입신호 없음
음식료,2022Q1,관찰,없음,확인된 보강 없음,True,고용 축 진입신호 없음
전기전자,2022Q1,관찰,없음,확인된 보강 없음,True,고용 축 진입신호 없음
철강,2022Q1,관찰,없음,확인된 보강 없음,True,고용 축 진입신호 없음


## 14. 분기별 단계 분포

In [11]:
display(HTML(delivery.distribution_html(read('stage_distribution_by_quarter.csv'))))
display(read('stage_distribution_by_industry.csv'))

industry,관찰,우선점검,추가확인
기계,9,9,0
기타,10,0,8
목재종이,10,2,6
비금속,12,0,6
석유화학,17,1,0
섬유의복,7,0,11
운송장비,17,0,1
음식료,13,5,0
전기전자,15,0,3
철강,18,0,0


## 15. 최신 10개 업종

같은 단계의 순위는 감소인원 기준이다. 관찰업종의 표시순서를 위험순위로 해석하지 않는다.

In [12]:
display(latest[['industry','quarter','decision_stage','q2_same_stage_rank','employment','emp_delta','signal_e','signal_r','signal_a','signal_p','decision_reason']].round(3))

industry,quarter,decision_stage,q2_same_stage_rank,employment,emp_delta,signal_e,signal_r,signal_a,signal_p,decision_reason
기계,2026Q2,우선점검,1,58784.0,-3979.0,6.340,2.704,3.465,19.547,고용감소 6.3%(진입경계 5% 통과) · 산단 제조업 고용의 3.47% 감소(상위경계 2.0% 통과) → 보강: 생산 19.5% 감소
목재종이,2026Q2,우선점검,2,428.0,-48.0,10.084,6.449,0.042,12.897,고용감소 10.1%(상위경계 10% 통과) · 산단평균 대비 6.4%p 열위(진입경계 통과) → 보강: 생산 12.9% 감소 / 직전 분기에도 진입신호(지속)
운송장비,2026Q2,관찰,1,16308.0,-129.0,0.785,0.000,0.112,0.000,고용 축 진입신호 없음
전기전자,2026Q2,관찰,2,27979.0,-116.0,0.413,0.000,0.101,0.000,고용 축 진입신호 없음
철강,2026Q2,관찰,3,9689.0,-112.0,1.143,0.000,0.098,0.000,고용 축 진입신호 없음
음식료,2026Q2,관찰,4,635.0,-25.0,3.788,0.152,0.022,0.000,고용 축 진입신호 없음
비금속,2026Q2,관찰,5,183.0,-2.0,1.081,0.000,0.002,0.000,고용 축 진입신호 없음
기타,2026Q2,관찰,6,171.0,25.0,0.000,0.000,0.000,0.000,고용 축 진입신호 없음
석유화학,2026Q2,관찰,7,619.0,54.0,0.000,0.000,0.000,6.730,고용 축 진입신호 없음 (생산 6.7% 단독 감소 — 확인질문에 반영)
섬유의복,2026Q2,관찰,8,34.0,0.0,0.000,0.000,0.000,12.320,고용 축 진입신호 없음 (생산 12.3% 단독 감소 — 확인질문에 반영)


## 16. 대표 진단카드 — 업종 분석을 현장 업무로 인계

구조화 카드 10개와 결정적 템플릿 진단문을 CSV/JSON/Markdown으로 생성했다. AI 추정 없이 실제 수치·질문을 전달한다. 담당자 검토·현장 근거·지원 검토결과는 미입력이다.

In [13]:
cards=read('diagnostic_cards_latest.csv')
display(Markdown(delivery.cards_markdown(cards[cards.industry.isin(['기계','목재종이'])])))

# 창원국가산단 산업·고용 전환진단 및 지원연계 모형 — 업종 진단카드

기존 산업동향 분석을 기업·현장 확인의 표준화된 진입점으로 전환한다.

산업동향 데이터 → Q1 상태 / Q2 규모 / Q3 시간 → 선제점검 판단 → 표준 업종 진단카드 → 담당자 검토 → 기업·현장 확인 → 기존 기업지원·고용지원·직업훈련 체계 검토 → 다음 분기 재점검

모형은 이번 분기에 어느 업종을 어느 수준으로 먼저 확인할지만 결정한다. 위기 여부·확률·원인·기업별 지원·사업 선정·예산은 결정하지 않는다.

## 기계 · 2026Q2 · 우선점검

- 같은 단계 처리순위: 1 (감소인원 순; 동률은 업종명 순)
- Q1: S4 생산↓·고용↓
- 관측사실: 명목 생산 YoY -19.55%; 고용 YoY -6.34%; 고용 증감 -3,979명; 산단 고용비중 51.19%
- Q2 전체 순증감 기여율: 91.85% (상쇄가 커 순변화/총변화 < 0.30이면 미표시)
- Q3 상태 지속: 2분기; 직전→현재 전환: S4 → S4; 반복 고용진입신호: False
- 최근 경로: 2025Q3:S3 → 2025Q4:S1 → 2026Q1:S4 → 2026Q2:S4
- E 6.34% / R 2.70%p / A 3.47% / P 19.55%
- 경계 통과: E True/False, R False/False, A True/True (진입/상위); P True; 규모 True
- 단계 근거: 고용감소 6.3%(진입경계 5% 통과) · 산단 제조업 고용의 3.47% 감소(상위경계 2.0% 통과) → 보강: 생산 19.5% 감소
- 자료 플래그: 고용핵심 미확인=False, 생산 미확인=False, 직전신호 미확인=False
- 집계자료로 알 수 없는 것: 수주·자동화·외주화·기업이전·고용조정 계획·미충원·설비전환의 유무와 원인은 집계자료만으로 확정할 수 없음
- 추가 확인질문: 수주잔량·가동률·휴업/감산·기업 수 변동·고용조정 계획 신고 여부를 확인한다(원인 후보이며 입증된 원인이 아님).
- 확인 담당 기능: 기업지원 기능 + 고용지원 기능
- 후속 연결: 현장 확인 결과에 따라 기존 기업지원·고용지원·직업훈련 체계 검토
- 담당자 검토: 미실시 / 현장 근거: 미입력 / 지원 검토결과: 미입력
- 다음 재점검: 2026Q3

## 목재종이 · 2026Q2 · 우선점검

- 같은 단계 처리순위: 2 (감소인원 순; 동률은 업종명 순)
- Q1: S4 생산↓·고용↓
- 관측사실: 명목 생산 YoY -12.90%; 고용 YoY -10.08%; 고용 증감 -48명; 산단 고용비중 0.37%
- Q2 전체 순증감 기여율: 1.11% (상쇄가 커 순변화/총변화 < 0.30이면 미표시)
- Q3 상태 지속: 2분기; 직전→현재 전환: S4 → S4; 반복 고용진입신호: True
- 최근 경로: 2025Q3:S1 → 2025Q4:S1 → 2026Q1:S4 → 2026Q2:S4
- E 10.08% / R 6.45%p / A 0.04% / P 12.90%
- 경계 통과: E True/True, R True/False, A False/False (진입/상위); P True; 규모 True
- 단계 근거: 고용감소 10.1%(상위경계 10% 통과) · 산단평균 대비 6.4%p 열위(진입경계 통과) → 보강: 생산 12.9% 감소 / 직전 분기에도 진입신호(지속)
- 자료 플래그: 고용핵심 미확인=False, 생산 미확인=False, 직전신호 미확인=False
- 집계자료로 알 수 없는 것: 수주·자동화·외주화·기업이전·고용조정 계획·미충원·설비전환의 유무와 원인은 집계자료만으로 확정할 수 없음
- 추가 확인질문: 수주잔량·가동률·휴업/감산·기업 수 변동·고용조정 계획 신고 여부를 확인한다(원인 후보이며 입증된 원인이 아님).
- 확인 담당 기능: 기업지원 기능 + 고용지원 기능
- 후속 연결: 현장 확인 결과에 따라 기존 기업지원·고용지원·직업훈련 체계 검토
- 담당자 검토: 미실시 / 현장 근거: 미입력 / 지원 검토결과: 미입력
- 다음 재점검: 2026Q3


## 17. 기존 ELECTRE/MRSort의 위치

초기 Q1~Q3 → ELECTRE 후보 구현 → 독립 감사(가중치·lambda·RC/VRC·VRC4·possible assignment·revision robustness) → 대안 비교 → 공공 선제대응 논리 조사 → 업종 단위 트리아지. 자유 선호공간에서 연구진 규범이 영향을 주는 점을 확인한 검증 이력이며 삭제하지 않는다.

**새 모형은 외부 기준과 명시된 운영규칙에 따라 하나의 행동단계를 반환하도록 재설계되었다.** 더 정확하다는 주장이 아니다. 대표값은 기존 독립 감사의 v1.0_crisp, possible set은 같은 감사의 연속공간 범위다.

In [14]:
comparison=read('comparison_vs_legacy.csv')
display(comparison[comparison.quarter==latest.quarter.iloc[0]][['industry','legacy_possible_set','legacy_representative','stage','changed_vs_representative','change_reason']])

industry,legacy_possible_set,legacy_representative,stage,changed_vs_representative,change_reason
기계,CHECK|PRIORITY,우선점검,우선점검,False,외부 논리·명시적 운영규칙의 단일 행동단계로 재설계; 고용감소 6.3%(진입경계 5% 통과) · 산단 제조업 고용의 3.47% 감소(상위경계 2.0% 통과) → 보강: 생산 19.5% 감소
기타,OBSERVE,관찰,관찰,False,외부 논리·명시적 운영규칙의 단일 행동단계로 재설계; 고용 축 진입신호 없음
목재종이,CHECK|PRIORITY,우선점검,우선점검,False,외부 논리·명시적 운영규칙의 단일 행동단계로 재설계; 고용감소 10.1%(상위경계 10% 통과) · 산단평균 대비 6.4%p 열위(진입경계 통과) → 보강: 생산 12.9% 감소 / 직전 분기에도 진입신호(지속)
비금속,OBSERVE|CHECK,관찰,관찰,False,외부 논리·명시적 운영규칙의 단일 행동단계로 재설계; 고용 축 진입신호 없음
석유화학,OBSERVE,관찰,관찰,False,외부 논리·명시적 운영규칙의 단일 행동단계로 재설계; 고용 축 진입신호 없음 (생산 6.7% 단독 감소 — 확인질문에 반영)
섬유의복,OBSERVE,관찰,관찰,False,외부 논리·명시적 운영규칙의 단일 행동단계로 재설계; 고용 축 진입신호 없음 (생산 12.3% 단독 감소 — 확인질문에 반영)
운송장비,OBSERVE|CHECK,추가확인,관찰,True,외부 논리·명시적 운영규칙의 단일 행동단계로 재설계; 고용 축 진입신호 없음
음식료,CHECK|PRIORITY,추가확인,관찰,True,외부 논리·명시적 운영규칙의 단일 행동단계로 재설계; 고용 축 진입신호 없음
전기전자,OBSERVE|CHECK,추가확인,관찰,True,외부 논리·명시적 운영규칙의 단일 행동단계로 재설계; 고용 축 진입신호 없음
철강,OBSERVE|CHECK,추가확인,관찰,True,외부 논리·명시적 운영규칙의 단일 행동단계로 재설계; 고용 축 진입신호 없음


### 초기 설계안의 채택 조건을 거친 현재 결론

|초기 구상|현재 위치|
|---|---|
|ELECTRE TRI-B 주모형 후보|구현·독립 감사한 비교모형|
|SMAA-TRI 안정성|파라미터 공간에 조건부인 후보 검증; 새 트리아지는 고정 시나리오 민감도|
|패널 고정효과 회귀|실행 근거 미확인, 수행 완료로 기재하지 않음|
|Isolation Forest|미수행 선택사항|
|Q1~Q3·진단카드|최종 핵심 흐름|

ELECTRE도 경계별 가중 지지도를 합산하므로 자의성이 자동 제거되는 것은 아니다. veto는 outranking 차단이며 심각한 신호의 자동 승격과 다르다. 표집 수용도는 위기 확률이나 연속공간 전체의 필연성이 아니다. 초기 g4(괴리 후보)와 실제 ELECTRE g4(고용 하회기간), 최종 반복신호는 서로 다르다.

[초기 문서의 항목별 정정과 팀원 공유 문장](../reports/model_role_clarification.md)

## 18. 최종 행정 연결과 재점검

진단카드 수신 → 담당자가 집계·개정·질문 검토 → 기업·현장 근거 확인 → 기업·경영 / 고용조정 / 인력·숙련 담당 기능에서 기존 체계 검토 → 판단·근거 기록 → 다음 분기 새 자료 재점검. 이는 제안 흐름이며 실제 기관 인계·지원 효과는 아직 검증하지 않았다.

In [15]:
display(HTML(delivery.flow_html()))

## 19. 프로젝트가 만드는 것

업종 단위 점검의 근거·순서·확인질문과 표준 인계카드를 만든다. 기존 행정체계의 기관·지원사업·예산을 새로 만들거나 자동 결정하지 않는다. 선택적 AI는 향후 이 카드의 문장 다듬기로만 제한할 수 있으며 현재 진단문은 결정적 템플릿이다.

## 20. 한계와 실행 근거

A·300인의 운영규범성, 일부 업종 편중, 사전등록 증거 부족, 명목 생산·집계업종 한계, 개정자료 의존, 현장 정답·연계효과 미검증이 남는다. 전체 감사는 reports/final_result_summary.md, 출처는 rule_provenance.csv와 institution_handoff_map.csv, 코드·원자료 해시는 run_metadata.json에서 확인한다. 노트북은 CSV와 Python 모듈을 표시하며 판정식을 복제하지 않는다.

In [16]:
display(json.loads((REPORT_ASSETS/'qa/triage_run_metadata.json').read_text(encoding='utf-8'))['rule_version'])
assert not any(c.endswith('_x') or c.endswith('_y') for c in panel.columns)
print('180행, 최신 10개 카드, 출처·민감도·인계 흐름 표시 완료')

'changwon-triage-rule/3.1.0-audited'

180행, 최신 10개 카드, 출처·민감도·인계 흐름 표시 완료


### 공식 출처와 상세 검증 파일

- [고용위기지역 고시 — 2026.5.4 시행](https://www.law.go.kr/LSW/admRulInfoP.do?admRulSeq=2100000278624&chrClsCd=010201)
- [지역 산업위기대응 고시 — 2025.3.4 시행](https://www.law.go.kr/LSW/admRulLsInfoP.do?admRulSeq=2100000255684)
- [최종 감사 보고서 A~N](../reports/final_result_summary.md)

triage_tests: tests=15, failures=0, errors=0, skipped=0; full_tests: tests=27, failures=0, errors=0, skipped=0

## 관찰

최신 분류와 근거 추적표를 함께 표시해 각 카드의 출처를 확인한다.

## 해석

우선순위는 행정적 확정이 아니라 기업·현장 확인 순서를 표준화하는 지원 정보다.

## 한계

실제 지원 가능성, 담당기관, 기업 상황은 인계 시점에 다시 확인해야 한다.

## Takeaway

핵심 신호부터 현장 확인 질문과 기존 기관 인계까지 한 흐름으로 제시한다.